In [2]:
import pandas as pd
import re
import os
import time

In [3]:
import cirpy

In [37]:
# reading S3 is chellenging, due to a format issue of the suppl.
# manually add them
roodt = [
   "Octanal", "1-Octanol/Octanol/Isooctanol", "1-Hexanol", 
    "2-ethyl-", "Heptanal", "2-Butenal", "(E)-", "7-Octen-2-ol", 
    "2,6-dimethyl-/Dihydromyrcenol", "Limonene", "Phenol", 
    "Benzyl alcohol", "Acetic acid/Ethanoic acid", "Hexanal", "Nonanal",
    "Decanal", "5-Hepten-2-one", "6-methyl-", 
    "6-Methyl-3,5-heptadiene-2-one", 
    "Geranylacetone/6,10-Dimethylundeca-5,9-dien-2-one", 
    "Dodecanoic acid", "Tetradecanoic acid/Myristic Acid", 
    "Pentadecanoic acid", "Heptadecanoic acid", "9,12-Octadecadienoic acid (Z,Z)-", 
    "2-Undecanone", "1-Nonanol", "Butyrolactone", 
    "Benzenemethanol", "α,α-dimethyl-", "Octanoic acid", 
    "Propanoic acid", "Butanoic acid", "Hexanoic acid", 
    "Heptanoic acid", "Nonanoic acid", "Decanoic acid", 
    "Tridecanoic acid", "Hexadecanoic acid/n-hexadecanoic acid", 
    "Octadecanoic acid", "Butanoic acid, 3-methyl-/Isovaleric acid", 
    "Hexanoic acid", "2-ethyl-", "2-Octenal", "(E)-", "2-Nonenal", "(E)-", 
    "2-Decenal", "(E)-2-Undecenal", "Furfural/2-Furancarboxyaldehyde", 
    "2-Furancarboxaldehyde", "5-methyl-", "2-Heptanone 2-Octanone", 
    "2-Nonanone", "2-Decanone", "2-Dodecanone", "2-Tridecanone", 
    "2-Pentadecanone", "2-Hexadecanone",
    "Acetophenone/1-phenyl-ethanone", "1-Hydroxy-2-butanone", 
    "Furyl hydroxymethyl ketone", "4H-Pyran-4-one",
    "2,3-dihydro-3,5-dihydroxy-6-methyl-", "Ethanone", 
    "1-(2-furanyl)-", "Ethanone", "1-(4-methylphenyl)-", "2-Undecanone",
    "6,10-dimethyl-", "2,3-Pentanedione", "Cyclopentanone", 
    "2-Cyclopenten-1-one", "Cyclopent-4-ene-1,3-dione", 
    "2-Cyclopenten-1-one", "2-methyl-", "2-Cyclopenten-1-one", 
    "3-methyl-", "1-Butanol", "2(3H)-Furanone", "5-methyl-", 
    "2(5H)-Furanone 2(3H)-Furanone", "5-butyldihydro-", 
    "2(3H)-Furanone", "5-hexyldihydro-", "2(3H)-Furanone", 
    "5-heptyldihydro-", "Linalool/3", "7- dimethyl-1,6-octadien-3-ol", 
    "α-Pinene", "Benzonitrile", "Pyrrole", "Pyridine Indole", 
    "3-methyl-", "5,10-Diethoxy-2,3,7,8-tetrahydro-1H,6H-dipyrrolo[1,2-α:1′,2′-δ]pyrazine", 
    "Dimethyl sulfone", "Dimethyl trisulphide", "1H-indole, Benzoic acid/Benzaldehyde", 
    "Benzeneacetaldehyde", "Phenol", "2-methyl-", "p-Cresol/4-methylphenol", 
    "Benzophenone", "Benzothiazole"]

wooding = [
    "Dodecanoic acid", "Tetradecanoic acid/Myristic Acid", 
    "Pentadecanoic acid", "Heptadecanoic acid 9,12-Octadecadienoic acid (Z,Z)-", 
    "Octanal", "2-Undecanone", "1-Octanol/Octanol/Isooctanol", 
    "1-Nonanol", "Octadecanol", "1-eicosanol", "ethanol", "2-(2-ethoxyethoxy)-", 
    "ethanol", "2-(dodecyloxy)-", "1", "octanol", "2-butyl", "1-heptanol", 
    "6-methyl-", "hexadecen-1-ol", "trans-9-", 
    "Butyrolactone,2,6,10,15,19,23-hexamethyl-2,6,10,14,18,22-tetracosahexaene (squalene)", 
    "cyclooctatetraene,naphthalene,5-ethyl-1,2,3,4—tetrahydro,cyclobutylamine", 
    "benzonitrile", "3,5-dimethyl-,pentadecane,2-methyl-hexadecane", "3-methyl", 
    "Hexadecane", "2,6,10,14-tetramethyl-", "Octane", "1,1’-oxybis-", 
    "4-cyanocyclohexane,Methyl salicylate", "acetic acid", 
    "octyl ester", "octadecanoicacid", "2,3, dihydroxypropyl ester", 
    "benzene butanoic acidgamma-oxo-", "ethyl ester", 
    "Isopropyl Palmitate 1,2-Benzenedicarboxylic acid", 
    "bis(2-methylpropyl) ester,octyl acrylate", "benzaldehyde", 
    "4-methyl-Benzenemethanol", "α,α-dimethyl-", "phenanthrene", "2,6-Diisopropylnaphthalene"
]

martin = [
    "Benzoic acid", "n- decanoic acid", "xylene (downregulated)", "3-carene (downregulated)"
]

compounds = roodt + wooding + martin
compounds = list(set(compounds))

In [38]:
len(compounds)

128

In [46]:
cass = {}

In [47]:
for i, compound in enumerate(compounds):
    time.sleep(1)
    try:
        out = cirpy.resolve(compound, "cas")
    except:
        out = None
    cass[compound] = out

In [63]:
# clean up list and unlist values
cass_re = {}
for key, values in cass.items():
    if values is None:
        cass_re[key] = values
    elif isinstance(values, list):
        cass_re[key] = ", ".join(values)
        
    else:
        cass_re[key] = values

In [65]:
pd.DataFrame({
    "Name": cass_re.keys(), "cas": cass_re.values()
}).to_excel("./results/summary_cas.xlsx")

In [ ]:
# Get chemical hierarchical names

In [14]:
# CAS  Cirpy
# input can be CASID: '288-36-8', or name "formic acid"
# output can be "stdinchikey", "cas"

# example
# cirpy.resolve("15907-03-6", "stdinchikey")

InChIKey=CSCPPACGZOOCGX-UHFFFAOYSA-N


In [13]:
# only S
#cas_list = [
#"2295-17-2", "624-83-9", "55771-40-9", "1009-61-6", "3796-70-1", "2761-24-2", "629-82-3", "6418-43-5", "36653-82-4", "75-15-0", "1795-09-1", "67875-54-1", "504-20-1", "540-88-5", "24050-09-7", "41446-67-7", "2765-11-9", "16747-33-4", "4584-63-8", "2460-77-7", "74645-98-0", "123-62-6", "1679-49-8", "636-41-9", "79-05-0", "56-89-3", "111-92-2", "35320-23-1", "1121-89-7", "664107", "54710-16-6", "62108-22-9", "4359-57-3", "2040-95-1", "35007-52-4", "592-84-7"
#]

# onlyM
cas_list = [
"111-14-8", "28564-83-2", "112-14-1", "79-09-4", "1002-84-2", "3913-02-8", "119-36-8", "87-64-9", "1604-28-0", "39546-75-3", "925-93-9", "18787-63-8", "65859-45-2", "124-07-2", "589-18-4", "118605-21-3", "112-05-0", "827-60-1", "100-47-0", "112-12-9", "80041-00-5", "57-11-4", "1560-92-5", "930-30-3", "80-56-8", "111-71-7", "24157-81-1", "20825-71-2", "1337-83-3", "68228-05-7", "65-85-0", "142-62-1", "13228-40-5", "13466-78-9", "108-97-4", "65104-67-8", "112-15-2", "6714-00-7", "111-27-3", "85-01-8", "2197-37-7", "4536-30-5", "78-70-6", "629-96-9", "5077-67-8", "2499-59-4", "119-61-9", "3658-80-8", "42775-75-7", "2548-87-0", "143-07-7", "60-33-3", "2463-53-8", "930-60-9", "171054-89-0", "4170-30-3", "5989-27-5", "593-08-8", "57-10-3", "706-14-9", "503-74-2", "18479-58-8", "2345-28-0", "141-78-6", "600-14-6", "544-63-8", "3913-81-3", "617-94-7", "64-17-5", "821-55-6", "104-67-6", "1438-92-2", "693-54-9", "37822-83-6", "107-92-6", "506-12-7", "1921-70-6", "1330-20-7", "120-92-3", "1216673-02-7"
]


In [14]:
out = []
for i in cas_list:
    time.sleep(1)
    out_= cirpy.resolve(i, 'stdinchikey')
    out.append(out_)

In [15]:
[re.sub("InChIKey=", "", i) if i is not None else "NA" for i in out]

['MNWFXJYAOYHMED-UHFFFAOYSA-N',
 'VOLMSPGWNYJHQQ-UHFFFAOYSA-N',
 'YLYBTZIQSIBWLI-UHFFFAOYSA-N',
 'XBDQKXXYIPTUBI-UHFFFAOYSA-N',
 'WQEPLUUGTLDZJY-UHFFFAOYSA-N',
 'XMVBHZBLHNOQON-UHFFFAOYSA-N',
 'OSWPMRLSEDHDFF-UHFFFAOYSA-N',
 'YPNZJHFXFVLXSE-UHFFFAOYSA-N',
 'KSKXSFZGARKWOW-GQCTYLIASA-N',
 'NA',
 'LFQSCWFLJHTTHZ-WFVSFCRTSA-N',
 'XCXKZBWAKKPFCJ-UHFFFAOYSA-N',
 'HQYSAIKKBLWFBW-UHFFFAOYSA-N',
 'WWZKQHOCKIZLMA-UHFFFAOYSA-N',
 'KMTDMTZBNYGUNX-UHFFFAOYSA-N',
 'NA',
 'FBUKVWPVBMHYJY-UHFFFAOYSA-N',
 'VYUNLDVEMSMJNP-UHFFFAOYSA-N',
 'JFDZBHWFFUWGJE-UHFFFAOYSA-N',
 'KYWIYKKSMDLRDC-UHFFFAOYSA-N',
 'NA',
 'QIQXTHQIDYTFRH-UHFFFAOYSA-N',
 'FNWWOHKUXFTKGN-UHFFFAOYSA-N',
 'BZKFMUIJRXWWQK-UHFFFAOYSA-N',
 'GRWFGVWFFZKLTI-UHFFFAOYSA-N',
 'FXHGMKSSBGDXIY-UHFFFAOYSA-N',
 'GWLLTEXUIOFAFE-UHFFFAOYSA-N',
 'RHDGNLCLDBVESU-UHFFFAOYSA-N',
 'PANBRUWVURLWGY-MDZDMXLPSA-N',
 'HNZUNIKWNYHEJJ-FMIVXFBMSA-N',
 'WPYMKLBDIGXBTP-UHFFFAOYSA-N',
 'FUZZWVXGSFPDMH-UHFFFAOYSA-N',
 'OLGGLCIDAMICTA-UHFFFAOYSA-N',
 'BQOFWKZOCNGFEC-UH